# Sales Analyst 1.5B Fine-Tune (QLoRA)

**Base model**: `Qwen/Qwen2.5-Coder-1.5B-Instruct`  
**Hardware**: Colab Pro, L4 GPU (~45 min)  
**Config**: `training/config/lora_1.5b.yaml`  
**Data**: `data/v1/train.jsonl`  
**Output**: `models/adapters/1.5b/`

Run cells top-to-bottom. Mount your Drive first so paths resolve.

In [ ]:
import os

# Install fine-tuning dependencies
# Let unsloth manage ALL ML library versions (peft, transformers, trl, accelerate, datasets, bitsandbytes)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q sentencepiece pyyaml

# Prevent CUDA OOM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import unsloth  # must be first — patches trl/transformers/peft before they load

import json
import os
import sys
from pathlib import Path

import yaml
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
REPO_ROOT   = Path("/content/salestools-analyst")
CONFIG_PATH = REPO_ROOT / "training/config/lora_1.5b.yaml"
PROMPT_PATH = REPO_ROOT / "training/config/system_prompt.txt"
TRAIN_DATA  = REPO_ROOT / "data/v1/train.jsonl"
ADAPTER_OUT = REPO_ROOT / "models/adapters/1.5b"
ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

# Clone repo if not already present
import subprocess, sys
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/sandeepkesarkar/salestools-analyst.git", str(REPO_ROOT)], check=True)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
SYSTEM_PROMPT = Path(PROMPT_PATH).read_text().strip()

print("Config:", cfg)
print(f"System prompt: {SYSTEM_PROMPT[:80]}...")

In [ ]:
REPO_ROOT   = Path("/content/salestools-analyst")
CONFIG_PATH = REPO_ROOT / "training/config/lora_1.5b.yaml"
PROMPT_PATH = REPO_ROOT / "training/config/system_prompt.txt"
TRAIN_DATA  = REPO_ROOT / "data/v1/train.jsonl"
ADAPTER_OUT = REPO_ROOT / "models/adapters/1.5b"

# Clone if config file not present (handles empty-dir edge case)
if not CONFIG_PATH.exists():
    import subprocess
    import shutil
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", "https://github.com/sandeepkesarkar/salestools-analyst.git", str(REPO_ROOT)], check=True)

ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
SYSTEM_PROMPT = Path(PROMPT_PATH).read_text().strip()

print("Config:", cfg)
print(f"System prompt: {SYSTEM_PROMPT[:80]}...")

In [ ]:
# ── Load base model with Unsloth ──────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg["base_model"],
    max_seq_length=cfg["max_seq_len"],
    dtype=None,           # auto-detect; bf16 on Ampere+
    load_in_4bit=True,    # QLoRA: 4-bit base
)

print("Base model loaded.")

In [ ]:
# ── Apply LoRA adapters ───────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg["lora_rank"],
    target_modules=cfg["target_modules"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=0,       # optimised
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg["seed"],
    use_rslora=False,
    loftq_config=None,
)

print("LoRA applied.")
model.print_trainable_parameters()

In [ ]:
# ── SFTTrainer setup ──────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=cfg["max_seq_len"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation"],
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=cfg["seed"],
        output_dir=str(ADAPTER_OUT / "checkpoints"),
        save_steps=100,
        save_total_limit=2,
        report_to="none",
    ),
)

print("Trainer ready.")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
trainer_stats = trainer.train()
print(f"Training complete. Runtime: {trainer_stats.metrics.get('train_runtime', 0):.0f}s")

In [ ]:
# ── Save LoRA adapter ─────────────────────────────────────────────────────────
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print(f"Adapter saved to {ADAPTER_OUT}")
print("Files:", list(ADAPTER_OUT.glob("*")))

## Next Steps

1. Download `models/adapters/1.5b/` from Drive to local machine
2. Run `bash training/export.sh` (set `MODEL_SIZE=1.5b ADAPTER_PATH=models/adapters/1.5b/`)
3. Verify: `ollama run sales-analyst-1.5b "Is my trend going up?"`
4. Eval: `python eval/run_eval.py --model sales-analyst-1.5b --held-out data/v1/held_out.jsonl`